In [1]:
import pandas as pd
import numpy as np
import os

## Read data

In [2]:
file_to_read = "SPY_60min.csv"
aggregate_timestamp = "60min"

print(os.getcwd())
df = pd.read_csv(f'../../../../worker/data/uncleaned/{file_to_read}')

c:\Users\owens\Desktop\PhoenixTrader\web-app\api\src\fetch\clean-data


In [3]:
df.head()

,timestamp,open,high,low,close,volume
0,2005-01-03 08:00:00,82.6786,82.6786,82.6242,82.6446,427400
1,2005-01-03 09:00:00,82.6378,82.7874,82.5766,82.6242,6192600
2,2005-01-03 10:00:00,82.6310,82.7058,82.0666,82.1618,14895400
3,2005-01-03 11:00:00,82.1618,82.2026,81.9239,82.1550,5683700
4,2005-01-03 12:00:00,82.1550,82.1618,81.9783,82.0599,2693500


## Parse Timestamps

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["timestamp"] = df["timestamp"].dt.tz_localize(
    "America/New_York",
    nonexistent="shift_forward",  # handles spring-forward DST
    ambiguous="NaT",  # handles fall-back DST
)
df["timestamp"] = df["timestamp"] + pd.Timedelta(hours=1)
df = df.set_index("timestamp").sort_index()

In [5]:
df.head()

,open,high,low,close,volume
timestamp,,,,,
2005-01-03 09:00:00-05:00,82.6786,82.6786,82.6242,82.6446,427400
2005-01-03 10:00:00-05:00,82.6378,82.7874,82.5766,82.6242,6192600
2005-01-03 11:00:00-05:00,82.6310,82.7058,82.0666,82.1618,14895400
2005-01-03 12:00:00-05:00,82.1618,82.2026,81.9239,82.1550,5683700
2005-01-03 13:00:00-05:00,82.1550,82.1618,81.9783,82.0599,2693500


## Remove After-Hours Data

In [6]:
market_open = "09:30"
market_close = "16:00"

df = df.between_time(market_open, market_close)

In [7]:
df.iloc[:10]

,open,high,low,close,volume
timestamp,,,,,
2005-01-03 10:00:00-05:00,82.6378,82.7874,82.5766,82.6242,6192600
2005-01-03 11:00:00-05:00,82.6310,82.7058,82.0666,82.1618,14895400
2005-01-03 12:00:00-05:00,82.1618,82.2026,81.9239,82.1550,5683700
2005-01-03 13:00:00-05:00,82.1550,82.1618,81.9783,82.0599,2693500
2005-01-03 14:00:00-05:00,82.0531,82.0599,81.7675,81.8083,4470600
2005-01-03 15:00:00-05:00,81.8083,81.9715,81.7539,81.7811,4794700
2005-01-03 16:00:00-05:00,81.7743,81.8695,81.5227,81.6315,11543400
2005-01-04 10:00:00-05:00,81.9171,81.9307,81.7335,81.7335,4140300
2005-01-04 11:00:00-05:00,81.7403,81.9579,81.5023,81.5635,9975600


## Fix Stock Splits

In [8]:
POSSIBLE_RATIOS = np.array([1 / 2, 1 / 3, 1 / 4, 1 / 5, 1 / 10, 2 / 1, 3 / 1, 4 / 1, 5 / 1])
TOLERANCE = 0.02 # 2%

def find_nearest_ratio(x):
    i = np.argmin(abs(POSSIBLE_RATIOS - x))
    if abs(POSSIBLE_RATIOS[i] - x) < TOLERANCE:
        return POSSIBLE_RATIOS[i]
    else:
        return np.nan


df["ratio"] = df["open"] / df["open"].shift(1)
df["split_ratio"] = df["ratio"].apply(find_nearest_ratio)

splits = df.dropna(subset=["split_ratio"])  # does not modify original df
if not splits.empty:
    print(f"Detected {len(splits)} split(s):")
    print(splits[["split_ratio"]])

# Adjust historical prices and volumes for each split
df_adjusted = df.copy()
for timestamp, row in splits.iterrows():
    ratio = row["split_ratio"]
    # All rows *before* the split timestamp need to be adjusted
    df_adjusted.loc[df_adjusted.index < timestamp, ["open", "high", "low", "close"]] *= ratio
    df_adjusted.loc[df_adjusted.index < timestamp, "volume"] /= ratio

# Clean up temporary columns
df = df_adjusted.drop(columns=["ratio", "split_ratio"], errors="ignore")

## Remove Bad Ticks

In [9]:
# Example thresholds
lower_thresh = 0.5  # 50% lower than neighbors
upper_thresh = 1.5  # 50% higher than neighbors

# Shifted neighbors
prev = df.shift(1)
next = df.shift(-1)

# Initialize mask for bad ticks
bad_mask = pd.Series(False, index=df.index)

# Check each OHLC column
for col in ["open", "high", "low", "close"]:
    bad_col = ((df[col] > prev[col] * upper_thresh) & (df[col] > next[col] * upper_thresh)) | \
              ((df[col] < prev[col] * lower_thresh) & (df[col] < next[col] * lower_thresh))
    bad_mask |= bad_col  # flag if any column is bad

# Logging
num_bad = bad_mask.sum()
print(f"Detected {num_bad} bad tick(s) out of {len(df)} rows")
if num_bad > 0:
    print("Timestamps of bad ticks:")
    print(df.index[bad_mask])

# Remove bad ticks
df = df[~bad_mask]

Detected 0 bad tick(s) out of 36688 rows


## Reindex Data

In [10]:
# Define constants
start_time = "10:00" if aggregate_timestamp == "60min" else market_open
freq = "1h" if aggregate_timestamp == "60min" else aggregate_timestamp

# Get unique dates
dates = df.index.normalize().unique()  # just the dates

# Build full RTH index
full_index = pd.DatetimeIndex(
    [ts for date in dates for ts in pd.date_range(
        start=f"{date.date()} {start_time}",
        end=f"{date.date()} {market_close}",
        freq=freq
    )]
)

# Assign timezone
full_index = full_index.tz_localize("America/New_York", ambiguous="NaT", nonexistent="shift_forward")

# Reindex only RTH
df = df.reindex(full_index)
df.iloc[:15]

,open,high,low,close,volume
2005-01-03 10:00:00-05:00,82.6378,82.7874,82.5766,82.6242,6192600.0
2005-01-03 11:00:00-05:00,82.6310,82.7058,82.0666,82.1618,14895400.0
2005-01-03 12:00:00-05:00,82.1618,82.2026,81.9239,82.1550,5683700.0
2005-01-03 13:00:00-05:00,82.1550,82.1618,81.9783,82.0599,2693500.0
2005-01-03 14:00:00-05:00,82.0531,82.0599,81.7675,81.8083,4470600.0
2005-01-03 15:00:00-05:00,81.8083,81.9715,81.7539,81.7811,4794700.0
2005-01-03 16:00:00-05:00,81.7743,81.8695,81.5227,81.6315,11543400.0
2005-01-04 10:00:00-05:00,81.9171,81.9307,81.7335,81.7335,4140300.0
2005-01-04 11:00:00-05:00,81.7403,81.9579,81.5023,81.5635,9975600.0
2005-01-04 12:00:00-05:00,81.5635,81.6587,81.4479,81.5363,7205300.0


## Linearly Interpolate Missing Data

In [11]:
# Boolean mask: True for rows with at least one missing value
rows_with_missing = df.isna().any(axis=1)

# Count how many rows have missing values
num_missing_rows = rows_with_missing.sum()

# Total number of rows
total_rows = len(df)

# Percentage of rows with missing values
percent_missing = 100 * num_missing_rows / total_rows

print(f"{num_missing_rows} out of {total_rows} rows are missing some value "
      f"({percent_missing:.2f}%)")

df = df.interpolate(method="linear")

6 out of 36694 rows are missing some value (0.02%)


In [12]:
df.head()

,open,high,low,close,volume
2005-01-03 10:00:00-05:00,82.6378,82.7874,82.5766,82.6242,6192600.0
2005-01-03 11:00:00-05:00,82.6310,82.7058,82.0666,82.1618,14895400.0
2005-01-03 12:00:00-05:00,82.1618,82.2026,81.9239,82.1550,5683700.0
2005-01-03 13:00:00-05:00,82.1550,82.1618,81.9783,82.0599,2693500.0
2005-01-03 14:00:00-05:00,82.0531,82.0599,81.7675,81.8083,4470600.0


## Front Fill and Back Fill

In [13]:
df = df.ffill().bfill()

## Save data to file

In [ ]:
# Use the same base directory as input, but for cleaned data
output_dir = (notebook_dir / '../../../../worker/data/cleaned').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

# Round to 4 decimals and integer volume
df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].round(4)
df["volume"] = df["volume"].round(0).astype(int)

# Save cleaned data
df.index.name = "timestamp"
output_path = output_dir / file_to_read
df.to_csv(output_path, index=True)
print(f"Saved cleaned data to: {output_path}")